In [1]:
import sys
sys.path.append('..')

from src.preprocess import preprocess
from src.features import select_features, engineer_features
from src.dimensionality import apply_pca, train_autoencoder
from src.models.random_forest import train_random_forest, evaluate as rf_evaluate, save_model as rf_save
from src.models.svm import train_svm, evaluate as svm_evaluate, save_model as svm_save
from src.models.lstm import train_lstm, evaluate as lstm_evaluate, save_model as lstm_save
from src.ensemble import run_ensemble
from src.evaluate import compute_metrics, plot_confusion_matrix, plot_training_history, compare_models

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('All imports OK')

All imports OK


In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Load only 2 files to keep memory manageable
files = [
    '../data/raw/cicids2017/Monday-WorkingHours.pcap_ISCX.csv',
    '../data/raw/cicids2017/Tuesday-WorkingHours.pcap_ISCX.csv',
]

dfs = []
for f in files:
    df = pd.read_csv(f, encoding='utf-8', low_memory=False)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    print(f"Loaded {os.path.basename(f)}: {df.shape}")
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

# Clean
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
print(f"\nCleaned shape: {df.shape}")

# Sample down to 200k rows to keep memory low
if len(df) > 200000:
    df = df.groupby('label', group_keys=False).apply(
        lambda x: x.sample(min(len(x), 20000), random_state=42)
    )
    print(f"Sampled shape: {df.shape}")

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
label_names = list(le.classes_)
num_classes = len(label_names)
print(f"\nClasses: {label_names}")

# Split
X = df.drop(columns=['label']).select_dtypes(include=[np.number])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"\nX_train: {X_train.shape}, X_test: {X_test.shape}")

Loaded Monday-WorkingHours.pcap_ISCX.csv: (529918, 79)
Loaded Tuesday-WorkingHours.pcap_ISCX.csv: (445909, 79)

Cleaned shape: (924226, 79)
Sampled shape: (29150, 79)

Classes: ['BENIGN', 'FTP-Patator', 'SSH-Patator']

X_train: (23320, 78), X_test: (5830, 78)


In [4]:
from src.dimensionality import apply_pca

X_train_pca, X_test_pca, pca = apply_pca(X_train, X_test, variance_threshold=0.95)
print(f'After PCA: {X_train_pca.shape}')

PCA: 78 features → 23 components (95% variance retained)
After PCA: (23320, 23)


In [5]:
rf_model = train_random_forest(X_train_pca, y_train)
rf_pred, rf_cm = rf_evaluate(rf_model, X_test_pca, y_test, label_names)
rf_metrics = compute_metrics(y_test, rf_pred, model_name='Random Forest')
plot_confusion_matrix(y_test, rf_pred, label_names, model_name='Random Forest')
rf_save(rf_model)

Training Random Forest...
Random Forest training complete.

--- Random Forest Results ---
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00      4000
 FTP-Patator       1.00      1.00      1.00      1186
 SSH-Patator       1.00      1.00      1.00       644

    accuracy                           1.00      5830
   macro avg       1.00      1.00      1.00      5830
weighted avg       1.00      1.00      1.00      5830

Confusion Matrix:
 [[4000    0    0]
 [   0 1183    3]
 [   1    0  643]]

[Random Forest] Accuracy: 0.9993 | F1: 0.9993 | Precision: 0.9993 | Recall: 0.9993 | FPR: 0.0004
Saved confusion matrix to results\cm_random_forest.png
Model saved to results/rf_model.pkl


In [6]:
svm_model = train_svm(X_train_pca, y_train)
svm_pred, svm_cm = svm_evaluate(svm_model, X_test_pca, y_test, label_names)
svm_metrics = compute_metrics(y_test, svm_pred, model_name='SVM')
plot_confusion_matrix(y_test, svm_pred, label_names, model_name='SVM')
svm_save(svm_model)

Training SVM...
SVM training complete.

--- SVM Results ---
              precision    recall  f1-score   support

      BENIGN       1.00      0.98      0.99      4000
 FTP-Patator       0.97      0.99      0.98      1186
 SSH-Patator       0.90      0.99      0.95       644

    accuracy                           0.98      5830
   macro avg       0.96      0.99      0.97      5830
weighted avg       0.98      0.98      0.98      5830

Confusion Matrix:
 [[3905   31   64]
 [   3 1180    3]
 [   3    3  638]]

[SVM] Accuracy: 0.9816 | F1: 0.9819 | Precision: 0.9828 | Recall: 0.9816 | FPR: 0.0078
Saved confusion matrix to results\cm_svm.png
Model saved to results/svm_model.pkl


In [7]:
lstm_model, history = train_lstm(X_train_pca, y_train, X_test_pca, y_test, num_classes)
lstm_pred, lstm_cm = lstm_evaluate(lstm_model, X_test_pca, y_test, label_names)
lstm_metrics = compute_metrics(y_test, lstm_pred, model_name='LSTM')
plot_confusion_matrix(y_test, lstm_pred, label_names, model_name='LSTM')
plot_training_history(history)
lstm_save(lstm_model)

Training LSTM...
Model: "lstm_ids"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 1, 128)            77824     
                                                                 
 dropout (Dropout)           (None, 1, 128)            0         
                                                                 
 lstm_1 (LSTM)               (None, 64)                49408     
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense (Dense)               (None, 64)                4160      
                                                                 
 dense_1 (Dense)             (None, 3)                 195       
                                                                 
Total params: 131587 (514.01 KB)
Trainabl

INFO:tensorflow:Assets written to: results/lstm_model\assets


Model saved to results/lstm_model


In [8]:
hard_pred, weighted_pred = run_ensemble(
    rf_model, svm_model, lstm_model,
    X_test_pca, y_test,
    weights=(0.3, 0.2, 0.5),
    label_names=label_names
)
hard_metrics     = compute_metrics(y_test, hard_pred,     model_name='Hard Vote')
weighted_metrics = compute_metrics(y_test, weighted_pred, model_name='Weighted Vote')

183/183 [==============================] - 1s 6ms/step

Running hard voting...

--- HARD VOTE Results ---
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00      4000
 FTP-Patator       1.00      0.99      1.00      1186
 SSH-Patator       0.99      0.99      0.99       644

    accuracy                           1.00      5830
   macro avg       1.00      1.00      1.00      5830
weighted avg       1.00      1.00      1.00      5830

Confusion Matrix:
 [[3996    3    1]
 [   3 1180    3]
 [   2    2  640]]

Running weighted voting...

--- WEIGHTED VOTE Results ---
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00      4000
 FTP-Patator       0.99      0.99      0.99      1186
 SSH-Patator       0.99      0.99      0.99       644

    accuracy                           1.00      5830
   macro avg       1.00      1.00      1.00      5830
weighted avg       1.00      1.00      1.00      5830


In [9]:
# Compare all models
all_metrics = [rf_metrics, svm_metrics, lstm_metrics, hard_metrics, weighted_metrics]
compare_models(all_metrics)

import pandas as pd
results_df = pd.DataFrame(all_metrics).set_index('model')
print(results_df)

# Save scaler, PCA and label encoder for the dashboard
import joblib, os
os.makedirs('results', exist_ok=True)
joblib.dump(scaler, '../results/scaler.pkl')
joblib.dump(pca,    '../results/pca.pkl')
joblib.dump(le,     '../results/label_encoder.pkl')
print('Scaler, PCA and label encoder saved.')

Saved model comparison chart to results\model_comparison.png
               accuracy  f1_score  precision  recall     fpr
model                                                       
Random Forest    0.9993    0.9993     0.9993  0.9993  0.0004
SVM              0.9816    0.9819     0.9828  0.9816  0.0078
LSTM             0.9969    0.9969     0.9969  0.9969  0.0023
Hard Vote        0.9976    0.9976     0.9976  0.9976  0.0015
Weighted Vote    0.9976    0.9976     0.9976  0.9976  0.0014
Scaler, PCA and label encoder saved.
